# Lab - Evaluate a Prompt Change on a Test Set

Three tasks that turn "this prompt looks better" into a number you can compare.

| Task | What you learn |
| --- | --- |
| 1 | A test set is questions plus the answers you expect |
| 2 | A small program scores a prompt against it |
| 3 | Same cases, two prompts, two scores |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `FUELIX_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"

# The first prompt: the role, the 30-day rule, and a verdict a program can read.
PROMPT_V1 = (
    "You are ShopAssist, an online store's support assistant. "
    "Returns are accepted within 30 days of delivery. "
    "Begin your answer with YES or NO, then one sentence."
)


# Provided: sends one question with one system prompt, returns the text block.
def answer(system, text):
    response = client.messages.create(
        model=MODEL,
        max_tokens=100,
        output_config={"effort": "low"},
        system=system,
        messages=[{"role": "user", "content": text}],
    )
    for block in response.content:
        if block.type == "text":
            return block.text

---

## Task 1 - Write the test set

In [ ]:
# ============================================================
# TASK 1 - Write the test set
# ============================================================
#
# WHAT TO DO
#   Six customer questions are written for you. Write the
#   verdict you expect for each one.
#
# WHY IT MATTERS
#   Every prompt edit changes some answers for the better and
#   some for the worse, and reading three replies by eye does
#   not tell you which. A test set does: questions, and for each
#   one the answer you committed to before anything ran.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Then the test set: six cases".
#
# HOW TO DO IT
#   Six blanks. Each verdict is "YES" or "NO", in quotes. The
#   store's rules:
#
#       returns are accepted within 30 days of delivery
#       gift cards cannot be returned
#       underwear cannot be returned
# ============================================================

# BEGIN SOLUTION
CASES = [
    ("I bought a jacket, delivered 20 days ago. Can I return it?", "YES"),
    ("The shoes arrived yesterday. I want to send them back.", "YES"),
    ("My order was delivered 45 days ago. Can I still return it?", "NO"),
    ("I got a gift card last week. Can I return it?", "NO"),
    ("The underwear I ordered came 3 days ago. Return possible?", "NO"),
    ("Delivered 29 days ago, is a return still possible?", "YES"),
]
# SCAFFOLD: CASES = [
# SCAFFOLD:     ("I bought a jacket, delivered 20 days ago. Can I return it?", ...),   # "YES" or "NO"?
# SCAFFOLD:     ("The shoes arrived yesterday. I want to send them back.", ...),
# SCAFFOLD:     ("My order was delivered 45 days ago. Can I still return it?", ...),
# SCAFFOLD:     ("I got a gift card last week. Can I return it?", ...),
# SCAFFOLD:     ("The underwear I ordered came 3 days ago. Return possible?", ...),
# SCAFFOLD:     ("Delivered 29 days ago, is a return still possible?", ...),
# SCAFFOLD: ]
# END SOLUTION: replace each ... below with "YES" or "NO"

for text, expected in CASES:
    print("{:<4} {}".format(str(expected), text))

check("test_set", cases=CASES)

---

## Task 2 - Score a prompt in code

In [ ]:
# ============================================================
# TASK 2 - Score a prompt in code
# ============================================================
#
# WHAT TO DO
#   Finish evaluate(): for each case, get the reply, check that
#   it starts with the expected verdict, count a pass, print the
#   case with PASS or FAIL - and return the score.
#
# WHY IT MATTERS
#   Six requests go out, one per case, and a program does the
#   reading. Look at the FAIL lines: the score is below the
#   maximum, and the failing case shows exactly where the prompt
#   has a gap.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now the two functions".
#
# HOW TO DO IT
#   Two blanks:
#
#       expected   the verdict from the test case - a reply
#                  passes when it starts with it
#       ok         True for a pass, False for a fail - adding it
#                  to score counts the passes
# ============================================================

# BEGIN SOLUTION
def evaluate(name, system):
    score = 0
    for text, expected in CASES:
        reply = answer(system, text)
        ok = reply.strip().upper().startswith(expected)
        score += ok
        print("PASS" if ok else "FAIL", "|", text[:45], "|", reply[:40])
    print(name, "score:", score, "/", len(CASES))
    return score
# SCAFFOLD: def evaluate(name, system):
# SCAFFOLD:     score = 0
# SCAFFOLD:     for text, expected in CASES:
# SCAFFOLD:         reply = answer(system, text)
# SCAFFOLD:         ok = reply.strip().upper().startswith(...)   # expected - the verdict from the case
# SCAFFOLD:         score += ...                                 # ok - True counts as 1
# SCAFFOLD:         print("PASS" if ok else "FAIL", "|", text[:45], "|", reply[:40])
# SCAFFOLD:     print(name, "score:", score, "/", len(CASES))
# SCAFFOLD:     return score
# END SOLUTION: replace each ... below with the value named beside it


try:
    v1_score = evaluate("v1", PROMPT_V1)
except TypeError as exc:   # provided: a blank left above, said plainly
    if "ellipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("score", evaluate=evaluate, v1_score=v1_score)

---

## Task 3 - Compare two prompts by their scores

In [ ]:
# ============================================================
# TASK 3 - Compare two prompts by their scores
# ============================================================
#
# WHAT TO DO
#   Write the second prompt - the first one plus one sentence -
#   and evaluate it on the same cases, right after the first.
#
# WHY IT MATTERS
#   Same cases, two prompts, two scores. If the second is
#   higher, the change was an improvement and you have a number
#   that says so, not a feeling. Six cases prove little on their
#   own: the value is in the comparison, and the set grows with
#   every real customer question that went wrong.
#
# WHERE TO SEE IT IN THE LECTURE
#   "So we write a second prompt".
#
# HOW TO DO IT
#   Two blanks:
#
#       NO_RETURN_RULE   the sentence the first prompt was missing
#                        (it is defined just below)
#       PROMPT_V2        the prompt to evaluate the second time
# ============================================================

NO_RETURN_RULE = " Gift cards and underwear cannot be returned."

try:
    # BEGIN SOLUTION
    PROMPT_V2 = PROMPT_V1 + NO_RETURN_RULE

    v2_score = evaluate("v2", PROMPT_V2)
    # SCAFFOLD: PROMPT_V2 = PROMPT_V1 + ...   # NO_RETURN_RULE - the missing sentence
    # SCAFFOLD:
    # SCAFFOLD: v2_score = evaluate("v2", ...)   # PROMPT_V2 - the prompt to evaluate now
    # END SOLUTION: replace each ... below with the value named beside it
except TypeError as exc:   # provided: a blank left above, said plainly
    if "ellipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

print()
print("v1:", v1_score, "/ 6    v2:", v2_score, "/ 6")

check("compare", v1_score=v1_score, v2_score=v2_score, prompt_v2=PROMPT_V2)

---

## Done

Keep a test set with expected answers, and never change a prompt without running it
before and after. The score is not the point; the comparison is.